# 02 Procesamiento de texto, recuperación secuencial e indexación

## Objetivo

Implementar y comparar dos mecanismos de recuperación de información —búsqueda secuencial e índice invertido— sobre colecciones de libros del Proyecto Gutenberg, investigando experimentalmente su rendimiento y escalabilidad.

### Parte A. Corpus pequeño

**1. Preparación del corpus**

Utilice los 18 documentos de la colección Gutenberg disponible en NLTK.

1. Cargue los textos originales.
2. Identifique el número de documentos y sus tamaños.
3. Implemente tokenización y normalización: conversión a minúsculas, tratamiento de puntuación y caracteres especiales.

Seleccione y documente una configuración de preprocesamiento que se utilizará consistentemente en todos los experimentos.

### Primero instalamos nltk y obtenemos los dastos

In [65]:
import re
import token

from PIL.ImagePalette import raw
%pip install nltk pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [66]:
import nltk
import pandas as pd
from nltk.corpus import gutenberg
nltk.download("gutenberg")

identificadores = gutenberg.fileids()
documentos_originales = {}
for identificador in identificadores:
    documentos_originales[identificador] = gutenberg.raw(identificador)
print(f"Número de documentos: {len(documentos_originales)}")

print("\nIdentificadores:")
for identificador in identificadores:
    print(identificador)



    primer_id = identificadores[0]

print(f"Documento: {primer_id}")
print("\nPrimeros 500 caracteres:")
print(documentos_originales[primer_id][:500])

Número de documentos: 18

Identificadores:
austen-emma.txt
austen-persuasion.txt
austen-sense.txt
bible-kjv.txt
blake-poems.txt
bryant-stories.txt
burgess-busterbrown.txt
carroll-alice.txt
chesterton-ball.txt
chesterton-brown.txt
chesterton-thursday.txt
edgeworth-parents.txt
melville-moby_dick.txt
milton-paradise.txt
shakespeare-caesar.txt
shakespeare-hamlet.txt
shakespeare-macbeth.txt
whitman-leaves.txt
Documento: austen-emma.txt

Primeros 500 caracteres:
[Emma by Jane Austen 1816]

VOLUME I

CHAPTER I


Emma Woodhouse, handsome, clever, and rich, with a comfortable home
and happy disposition, seemed to unite some of the best blessings
of existence; and had lived nearly twenty-one years in the world
with very little to distress or vex her.

She was the youngest of the two daughters of a most affectionate,
indulgent father; and had, in consequence of her sister's marriage,
been mistress of his house from a very early period.  Her mother
had died t


[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


### implementamos el tratamiento de los datos

In [67]:
documentos_tokens = {}
documentos_procesados = {}
def preprocesar(texto):
    texto = texto.lower()
    tokens = re.findall(r"[^\W\d_]+", texto)
    return tokens

for identificador in identificadores:
    texto_original = documentos_originales[identificador]

    tokens = preprocesar(texto_original)
    documentos_tokens[identificador] = tokens
    documentos_procesados[identificador] = " ".join(tokens)
print(f"Documentos procesados: {len(documentos_procesados)}")
primer_id = identificadores[0]
print(f"\nDocumento: {primer_id}")
print("\nPrimeros 30 tokens:")
print(documentos_tokens[primer_id][:30])

Documentos procesados: 18

Documento: austen-emma.txt

Primeros 30 tokens:
['emma', 'by', 'jane', 'austen', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich', 'with', 'a', 'comfortable', 'home', 'and', 'happy', 'disposition', 'seemed', 'to', 'unite', 'some', 'of', 'the', 'best', 'blessings', 'of']


**2. Búsqueda secuencial**

Implemente una función que reciba un término de consulta y devuelva los identificadores de todos los documentos que lo contienen con una búsqueda secuencial.

La implementación deberá utilizar obligatoriamente dos lazos `for` anidados: el primero recorre los documentos y el segundo los tokens de cada documento.

No es necesario utilizar índices invertidos, motores de búsqueda ni estructuras auxiliares que almacenen anticipadamente los documentos asociados a cada término.

Ejecute consultas con términos frecuentes, poco frecuentes e inexistentes.

In [68]:
def busqueda_secuencial(termino, documentos_tokens):
    tokens_consulta = preprocesar(termino)
    if len(tokens_consulta) != 1:
        raise ValueError(
            "La consulta debe contener exactamente un término "
            "después del preprocesamiento."
        )

    termino_normalizado = tokens_consulta[0]
    resultados = []

    for identificador, tokens in documentos_tokens.items():
        for token in tokens:

            if token == termino_normalizado:
                resultados.append(identificador)
                break

    return resultados

### Búsqueda de ejemplo

In [69]:
consultas = {
    "Frecuente": "the",
    "Poco frecuente": "whale",
    "Inexistente": "xyzabc"
}
for categoria, termino in consultas.items():

    resultados = busqueda_secuencial(
        termino,
        documentos_tokens
    )
    print(f"\nCategoría: {categoria}")
    print(f"Término: {termino}")
    print(f"Documentos encontrados: {len(resultados)}")
    for identificador in resultados:
        print(f"  - {identificador}")


Categoría: Frecuente
Término: the
Documentos encontrados: 18
  - austen-emma.txt
  - austen-persuasion.txt
  - austen-sense.txt
  - bible-kjv.txt
  - blake-poems.txt
  - bryant-stories.txt
  - burgess-busterbrown.txt
  - carroll-alice.txt
  - chesterton-ball.txt
  - chesterton-brown.txt
  - chesterton-thursday.txt
  - edgeworth-parents.txt
  - melville-moby_dick.txt
  - milton-paradise.txt
  - shakespeare-caesar.txt
  - shakespeare-hamlet.txt
  - shakespeare-macbeth.txt
  - whitman-leaves.txt

Categoría: Poco frecuente
Término: whale
Documentos encontrados: 6
  - bible-kjv.txt
  - bryant-stories.txt
  - chesterton-ball.txt
  - melville-moby_dick.txt
  - shakespeare-hamlet.txt
  - whitman-leaves.txt

Categoría: Inexistente
Término: xyzabc
Documentos encontrados: 0


**3. Medición de tiempos de ejecución**

Utilice `time.perf_counter_ns()` o una herramienta de medición equivalente para obtener el tiempo de ejecución de las consultas.

Cada medición deberá corresponder a **30 ejecuciones de la misma consulta**. Y se debe calcular la media aritmética.

Registre los resultados en milisegundos. No incluya la carga de archivos ni la tokenización dentro del tiempo de consulta.

In [70]:
import time
import statistics
import pandas as pd
def buscar_termino_normalizado(termino, documentos_tokens):
    resultados = []
    for identificador, tokens in documentos_tokens.items():
        for token in tokens:

            if token == termino:
                resultados.append(identificador)
                break

    return resultados

def medir_busqueda(consulta, documentos_tokens, repeticiones=30):
    tokens_consulta = preprocesar(consulta)

    if len(tokens_consulta) != 1:
        raise ValueError(
            "La consulta debe contener exactamente un término "
            "después del preprocesamiento."
        )
    termino = tokens_consulta[0]
    tiempos_ms = []
    for _ in range(repeticiones):
        inicio = time.perf_counter_ns()
        resultados = buscar_termino_normalizado(
            termino,
            documentos_tokens
        )
        fin = time.perf_counter_ns()
        tiempo_ms = (fin - inicio) / 1_000_000
        tiempos_ms.append(tiempo_ms)
    media_ms = statistics.mean(tiempos_ms)
    return resultados, tiempos_ms, media_ms

consultas = {
    "Frecuente": "the",
    "Poco frecuente": "whale",
    "Inexistente": "xyzabc"
}

resumen = []
tiempos_por_consulta = {}

for categoria, consulta in consultas.items():

    resultados, tiempos, media = medir_busqueda(
        consulta,
        documentos_tokens,
        repeticiones=30
    )
    tiempos_por_consulta[consulta] = tiempos
    resumen.append({
        "Categoría": categoria,
        "Consulta": consulta,
        "Documentos encontrados": len(resultados),
        "Ejecuciones": len(tiempos),
        "Tiempo promedio (ms)": media
    })
tabla_tiempos = pd.DataFrame(resumen)
display(tabla_tiempos)

,Categoría,Consulta,Documentos encontrados,Ejecuciones,Tiempo promedio (ms)
0,Frecuente,the,18,30,0.010540
1,Poco frecuente,whale,6,30,38.278673
2,Inexistente,xyzabc,0,30,60.086237


**4. Construcción del índice invertido**

Implemente un índice invertido que contenga el vocabulario de la colección y las listas de documentos asociados a cada término.

Inspeccione las listas de postings y calcule la frecuencia documental (DF) de varios términos.

Implemente consultas de un término y consultas booleanas con AND, OR y NOT.

### Creación del índice invertido

In [71]:
def construir_indice_invertido(documentos_tokens):
    indice = {}
    for identificador, tokens in documentos_tokens.items():
        for token in tokens:
            if token not in indice:
                indice[token] = set()
            indice[token].add(identificador)
    return indice
indice_invertido = construir_indice_invertido(documentos_tokens)
print(f"Número de documentos: {len(documentos_tokens)}")
print(f"Tamaño del vocabulario: {len(indice_invertido)}")

Número de documentos: 18
Tamaño del vocabulario: 41509


### Inspeccionamos la lista de postings y frecuencia documental

In [72]:
cantidad = 5
for termino in list(indice_invertido.keys())[:cantidad]:
    print(f"\nTérmino: {termino}")
    print(f"Postings: {sorted(indice_invertido[termino])}")
def frecuencia_documental(termino,indice_invertido):
    tokens = preprocesar(termino)
    if len(tokens) != 1:
        raise ValueError(
            "La consulta debe contener exactamente un término "
            "después del preprocesamiento."
        )
    termino_normalizado = tokens[0]
    postings = indice_invertido.get(termino_normalizado, set())
    return len(postings)
terminos = ["the", "whale", "leviathan", "xyzabc"]
for termino in terminos:
    df = frecuencia_documental(termino, indice_invertido)
    print(f"Término: {termino}")
    print(f"Frecuencia documental: {df}")



Término: emma
Postings: ['austen-emma.txt', 'austen-persuasion.txt']

Término: by
Postings: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']

Término: jane
Postings: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt']

Término: austen
Postings: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt']

Término: volume
Postings: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'melville-moby_dick.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
Término: the
Frecuenc

### Consultas de un término

In [73]:
def buscar_termino(consulta, indice_invertido):
    tokens = preprocesar(consulta)
    if len(tokens) != 1:
        raise ValueError("La consulta debe contener un solo término.")
    termino = tokens[0]
    return indice_invertido.get(termino, set()).copy()
resultado = buscar_termino("Whale", indice_invertido)
print(f"Documentos encontrados: {len(resultado)}")
for documento in sorted(resultado):
    print(documento)

Documentos encontrados: 6
bible-kjv.txt
bryant-stories.txt
chesterton-ball.txt
melville-moby_dick.txt
shakespeare-hamlet.txt
whitman-leaves.txt


### Consultas booleanas

In [74]:
def consulta_booleana(operador, termino_1, indice_invertido,documentos_tokens,termino_2=None):
    operador = operador.upper()

    documentos_1 = buscar_termino(termino_1, indice_invertido)

    if operador == "NOT":
        todos_los_documentos = set(documentos_tokens.keys())
        return todos_los_documentos - documentos_1

    if operador in ("AND", "OR"):
        if termino_2 is None:
            raise ValueError(f"El operador {operador} necesita dos términos.")

        documentos_2 = buscar_termino(termino_2, indice_invertido)

        if operador == "AND":
            return documentos_1 & documentos_2

        return documentos_1 | documentos_2

    raise ValueError("Operador no válido. Usa AND, OR o NOT.")

In [75]:
resultado_and = consulta_booleana(
    "AND", "whale", indice_invertido, documentos_tokens, "sea"
)
print("WH ALE AND SEA:", sorted(resultado_and))
print("Total:", len(resultado_and))
resultado_or = consulta_booleana(
    "OR", "whale", indice_invertido, documentos_tokens, "sea"
)
print("WHALE OR SEA:", sorted(resultado_or))
print("Total:", len(resultado_or))
resultado_not = consulta_booleana(
    "NOT", "whale", indice_invertido, documentos_tokens
)
print("NOT WHALE:", sorted(resultado_not))
print("Total:", len(resultado_not))

WH ALE AND SEA: ['bible-kjv.txt', 'bryant-stories.txt', 'chesterton-ball.txt', 'melville-moby_dick.txt', 'shakespeare-hamlet.txt', 'whitman-leaves.txt']
Total: 6
WHALE OR SEA: ['austen-emma.txt', 'austen-persuasion.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
Total: 16
NOT WHALE: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'blake-poems.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-macbeth.txt']
Total: 12


**5. Comparación**

Ejecute las mismas consultas con ambos métodos.

Compruebe que recuperan exactamente los mismos identificadores de documentos y mida el tiempo medio de 30 ejecuciones para cada consulta y método.

In [76]:
consultas_un_termino = {
    "Frecuente: the": "the",
    "Poco frecuente: whale": "whale",
    "Inexistente: xyzabc": "xyzabc"
}

consultas_booleanas = [
    ("whale AND sea", "AND", "whale", "sea"),
    ("whale OR sea", "OR", "whale", "sea"),
    ("NOT whale", "NOT", "whale", None)
]

universo = set(documentos_tokens.keys())
resumen = []
tiempos_por_consulta = {}

# comparar consultas de un término
for nombre, consulta in consultas_un_termino.items():
    resultados_sec, tiempos_sec, media_sec = medir_busqueda(
        consulta,
        documentos_tokens,
        repeticiones=30
    )
    tokens_consulta = preprocesar(consulta)
    if len(tokens_consulta) != 1:
        raise ValueError(
            f"La consulta '{consulta}' debe contener un solo término."
        )

    termino = tokens_consulta[0]
    tiempos_ind = []
    for _ in range(30):
        inicio = time.perf_counter_ns()

        resultados_ind = indice_invertido.get(termino, set()).copy()

        fin = time.perf_counter_ns()
        tiempos_ind.append((fin - inicio) / 1_000_000)

    media_ind = statistics.mean(tiempos_ind)
    resultados_sec = set(resultados_sec)
    coinciden = resultados_sec == resultados_ind
    tiempos_por_consulta[nombre] = {
        "secuencial": tiempos_sec,
        "indice": tiempos_ind
    }

    resumen.append({
        "Consulta": nombre,
        "Documentos encontrados": len(resultados_sec),
        "Resultados iguales": coinciden,
        "Ejecuciones por método": 30,
        "Media secuencial (ms)": media_sec,
        "Media índice (ms)": media_ind
    })

    if not coinciden:
        print(f"\nDiferencias en: {nombre}")
        print("Solo en secuencial:", sorted(resultados_sec - resultados_ind))
        print("Solo en índice:", sorted(resultados_ind - resultados_sec))
#  comparar consultas booleanas

for nombre, operador, consulta_1, consulta_2 in consultas_booleanas:
    tokens_1 = preprocesar(consulta_1)

    if len(tokens_1) != 1:
        raise ValueError(
            f"'{consulta_1}' debe producir exactamente un término."
        )

    termino_1 = tokens_1[0]
    if operador in ("AND", "OR"):
        tokens_2 = preprocesar(consulta_2)

        if len(tokens_2) != 1:
            raise ValueError(
                f"'{consulta_2}' debe producir exactamente un término."
            )

        termino_2 = tokens_2[0]
    tiempos_sec = []

    for _ in range(30):
        inicio = time.perf_counter_ns()
        documentos_1 = set(
            buscar_termino_normalizado(
                termino_1,
                documentos_tokens
            )
        )

        if operador == "NOT":
            resultados_sec = universo - documentos_1

        else:
            documentos_2 = set(
                buscar_termino_normalizado(
                    termino_2,
                    documentos_tokens
                )
            )

            if operador == "AND":
                resultados_sec = documentos_1 & documentos_2
            elif operador == "OR":
                resultados_sec = documentos_1 | documentos_2
        fin = time.perf_counter_ns()
        tiempos_sec.append((fin - inicio) / 1_000_000)
    media_sec = statistics.mean(tiempos_sec)
    # índice invertido
    tiempos_ind = []

    for _ in range(30):
        inicio = time.perf_counter_ns()

        documentos_1 = indice_invertido.get(termino_1, set()).copy()

        if operador == "NOT":
            resultados_ind = universo - documentos_1

        else:
            documentos_2 = indice_invertido.get(termino_2, set()).copy()

            if operador == "AND":
                resultados_ind = documentos_1 & documentos_2
            elif operador == "OR":
                resultados_ind = documentos_1 | documentos_2

        fin = time.perf_counter_ns()
        tiempos_ind.append((fin - inicio) / 1_000_000)

    media_ind = statistics.mean(tiempos_ind)
    # resultados

    coinciden = resultados_sec == resultados_ind
    tiempos_por_consulta[nombre] = {
        "secuencial": tiempos_sec,
        "indice": tiempos_ind
    }

    resumen.append({
        "Consulta": nombre,
        "Documentos encontrados": len(resultados_sec),
        "Resultados iguales": coinciden,
        "Ejecuciones por método": 30,
        "Media secuencial (ms)": media_sec,
        "Media índice (ms)": media_ind
    })

    if not coinciden:
        print(f"\nDiferencias en: {nombre}")
        print("Solo en secuencial:", sorted(resultados_sec - resultados_ind))
        print("Solo en índice:", sorted(resultados_ind - resultados_sec))
tabla_comparacion = pd.DataFrame(resumen)

display(tabla_comparacion)

assert tabla_comparacion["Resultados iguales"].all(), (
    "Hay consultas cuyos resultados no coinciden."
)

,Consulta,Documentos encontrados,Resultados iguales,Ejecuciones por método,Media secuencial (ms),Media índice (ms)
0,Frecuente: the,18,True,30,0.009243,0.000817
1,Poco frecuente: whale,6,True,30,37.578940,0.000660
2,Inexistente: xyzabc,0,True,30,63.697890,0.000757
3,whale AND sea,6,True,30,46.451337,0.005423
4,whale OR sea,16,True,30,44.296383,0.004810
5,NOT whale,12,True,30,42.631337,0.001507


### Parte B. Corpus grande

**6. Descarga de Gutenberg**

Descargue el archivo oficial `txt-files.tar.zip`, de aproximadamente 11 GB, desde:

https://www.gutenberg.org/cache/epub/feeds/

Identifique los archivos TXT, evite contar distintas versiones del mismo libro como documentos independientes y registre la cantidad de documentos válidos.

In [77]:
from pathlib import Path
from urllib.request import urlretrieve
carpeta = Path(r"C:\Users\ASUS\Documents\gutenberg_grande")
carpeta.mkdir(parents=True, exist_ok=True)
url = "https://www.gutenberg.org/cache/epub/feeds/txt-files.tar.zip"
archivo_zip = carpeta / "txt-files.tar.zip"
if archivo_zip.exists():
    print(f"El archivo ya existe: {archivo_zip}")
else:
    print("Descargando el corpus. Esto puede tardar bastante...")

    try:
        urlretrieve(url, archivo_zip)
        print("Descarga completada.")
    except Exception as error:
        archivo_zip.unlink(missing_ok=True)
        print(f"No se completó la descarga: {error}")

El archivo ya existe: C:\Users\ASUS\Documents\gutenberg_grande\txt-files.tar.zip


In [78]:
from pathlib import Path
import subprocess

carpeta = Path(r"C:\Users\ASUS\Documents\gutenberg_grande")
carpeta.mkdir(parents=True, exist_ok=True)

url = "https://www.gutenberg.org/cache/epub/feeds/txt-files.tar.zip"
archivo_zip = carpeta / "txt-files.tar.zip"

resultado = subprocess.run(
    [
        "curl.exe",
        "-L",
        "-C", "-",               # Continuar desde el tamaño actual
        "--retry", "10",         # Reintentar errores temporales
        "--retry-delay", "10",   # Esperar 10 segundos entre reintentos
        "-o", str(archivo_zip),
        url
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

if archivo_zip.exists():
    tamano = archivo_zip.stat().st_size / (1024 ** 3)
    print(f"Tamaño descargado: {tamano:.2f} GiB")

print("Código de salida:", resultado.returncode)

if resultado.returncode == 0:
    print("Descarga finalizada.")
else:
    print("\nÚltimas líneas del mensaje de curl:")
    print("\n".join(resultado.stderr.splitlines()[-8:]))

Tamaño descargado: 8.86 GiB
Código de salida: 56

Últimas líneas del mensaje de curl:
 68  5.41G  68  3.71G   0      0  6.89M      0   13:23   09:11   04:12 132.3k
 68  5.41G  68  3.71G   0      0  6.87M      0   13:25   09:12   04:13  69335
 68  5.41G  68  3.71G   0      0  6.86M      0   13:26   09:13   04:13  91813
 68  5.41G  68  3.71G   0      0  6.85M      0   13:28   09:14   04:14 118.1k
 68  5.41G  68  3.71G   0      0  6.84M      0   13:29   09:15   04:14 177.7k
 68  5.41G  68  3.71G   0      0  6.83M      0   13:31   09:16   04:15 249.7k
 68  5.41G  68  3.71G   0      0  6.82M      0   13:32   09:17   04:15 336.7k
curl: (56) schannel: server closed abruptly (missing close_notify)


In [83]:
from pathlib import Path
import subprocess
import zipfile

carpeta = Path(r"C:\Users\ASUS\Documents\gutenberg_grande")
archivo_zip = carpeta / "txt-files.tar.zip"

url = "https://www.gutenberg.org/cache/epub/feeds/txt-files.tar.zip"

print("Intentando continuar la descarga...", flush=True)

resultado = subprocess.run(
    [
        "curl.exe",
        "-L",
        "-C", "-",
        "--retry", "10",
        "--retry-delay", "10",
        "-o", str(archivo_zip),
        url
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

tamano_gib = archivo_zip.stat().st_size / (1024 ** 3)

print(f"Tamaño actual: {tamano_gib:.2f} GiB")
print("Código de salida:", resultado.returncode)
print("¿Python reconoce el ZIP?:", zipfile.is_zipfile(archivo_zip))

if resultado.returncode != 0:
    print("\nMensaje de curl:")
    print("\n".join(resultado.stderr.splitlines()[-5:]))

Intentando continuar la descarga...
Tamaño actual: 10.56 GiB
Código de salida: 0
¿Python reconoce el ZIP?: True


In [86]:
import zipfile
import tarfile

print("Leyendo los nombres dentro del archivo...", flush=True)

with zipfile.ZipFile(archivo_zip, "r") as zip_corpus:

    # Abrir el TAR directamente desde el ZIP
    with zip_corpus.open("txt-files.tar") as archivo_tar:

        # Modo de lectura secuencial: no extrae el TAR al disco
        with tarfile.open(fileobj=archivo_tar, mode="r|*") as tar_corpus:

            cantidad = 0

            for miembro in tar_corpus:

                if miembro.isfile():
                    print(miembro.name)

                    cantidad += 1

                    if cantidad == 20:
                        break

print("\nLectura de ejemplo terminada.")

Leyendo los nombres dentro del archivo...
cache/epub/10000/pg10000.txt
cache/epub/10001/pg10001.txt
cache/epub/10002/pg10002.txt
cache/epub/10003/pg10003.txt
cache/epub/10004/pg10004.txt
cache/epub/10005/pg10005.txt
cache/epub/10006/pg10006.txt
cache/epub/10007/pg10007.txt
cache/epub/10008/pg10008.txt
cache/epub/10009/pg10009.txt
cache/epub/1000/pg1000.txt
cache/epub/10010/pg10010.txt
cache/epub/10011/pg10011.txt
cache/epub/10012/pg10012.txt
cache/epub/10013/pg10013.txt
cache/epub/10014/pg10014.txt
cache/epub/10015/pg10015.txt
cache/epub/10016/pg10016.txt
cache/epub/10017/pg10017.txt
cache/epub/10018/pg10018.txt

Lectura de ejemplo terminada.


In [88]:
from pathlib import Path
import zipfile
import shutil

carpeta = Path(r"C:\Users\ASUS\Documents\gutenberg_grande")
archivo_zip = carpeta / "txt-files.tar.zip"

archivo_tar = carpeta / "txt-files.tar"
archivo_temporal = carpeta / "txt-files.tar.parcial"

with zipfile.ZipFile(archivo_zip, "r") as zip_corpus:

    informacion = zip_corpus.getinfo("txt-files.tar")
    tamano_tar = informacion.file_size

    print(f"Tamaño del TAR: {tamano_tar / (1024 ** 3):.2f} GiB")

    # Si ya existe un TAR del tamaño esperado, no lo extraemos de nuevo
    if archivo_tar.exists() and archivo_tar.stat().st_size == tamano_tar:

        print("El TAR ya está extraído:", archivo_tar)

    else:

        espacio_libre = shutil.disk_usage(carpeta).free

        print(f"Espacio libre: {espacio_libre / (1024 ** 3):.2f} GiB")

        if espacio_libre < tamano_tar:
            raise OSError(
                "No hay suficiente espacio para extraer el TAR completo."
            )

        print("Extrayendo el TAR. Esto puede tardar varios minutos...", flush=True)

        with zip_corpus.open("txt-files.tar") as origen:
            with open(archivo_temporal, "wb") as destino:
                shutil.copyfileobj(origen, destino, length=1024 * 1024)

        if archivo_temporal.stat().st_size != tamano_tar:
            raise IOError("La extracción quedó incompleta.")

        archivo_temporal.replace(archivo_tar)

        print("Extracción terminada:", archivo_tar)

Tamaño del TAR: 28.55 GiB
Espacio libre: 46.74 GiB
Extrayendo el TAR. Esto puede tardar varios minutos...
Extracción terminada: C:\Users\ASUS\Documents\gutenberg_grande\txt-files.tar


In [90]:
import re
import tarfile
from pathlib import Path

archivo_tar = Path(
    r"C:\Users\ASUS\Documents\gutenberg_grande\txt-files.tar"
)

if not archivo_tar.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {archivo_tar}")

# Aquí guardaremos los tokens de los libros seleccionados
documentos_tokens_grandes = {}

# Para contar identificadores únicos en todo el TAR
ids_reconocidos = set()

archivos_txt = 0
archivos_no_reconocidos = 0
archivos_vacios = 0
errores_procesamiento = 0

print("Leyendo el corpus grande...", flush=True)

with tarfile.open(archivo_tar, "r:") as tar_corpus:

    for miembro in tar_corpus:

        if not miembro.isfile():
            continue

        if not miembro.name.lower().endswith(".txt"):
            continue

        archivos_txt += 1

        # Ejemplo: cache/epub/10009/pg10009.txt
        coincidencia = re.fullmatch(
            r"cache/epub/(\d+)/pg(\d+)\.txt",
            miembro.name,
            flags=re.IGNORECASE
        )

        if coincidencia is None:
            archivos_no_reconocidos += 1
            continue

        id_carpeta = coincidencia.group(1)
        id_archivo = coincidencia.group(2)

        if id_carpeta != id_archivo:
            archivos_no_reconocidos += 1
            continue

        if miembro.size == 0:
            archivos_vacios += 1
            continue

        # Contar cada ID una sola vez
        if id_carpeta in ids_reconocidos:
            continue

        ids_reconocidos.add(id_carpeta)

        # Solo procesar los primeros 1.000 libros que se puedan leer
        if len(documentos_tokens_grandes) < 1000:

            archivo_libro = tar_corpus.extractfile(miembro)

            if archivo_libro is None:
                errores_procesamiento += 1
                continue

            with archivo_libro:
                contenido = archivo_libro.read()

            try:
                texto = contenido.decode("utf-8-sig")
            except UnicodeDecodeError:
                errores_procesamiento += 1
                continue

            tokens = preprocesar(texto)

            if not tokens:
                errores_procesamiento += 1
                continue

            documentos_tokens_grandes[id_carpeta] = tokens

            if len(documentos_tokens_grandes) % 100 == 0:
                print(
                    f"Documentos procesados: "
                    f"{len(documentos_tokens_grandes)}",
                    flush=True
                )

print("\nLectura terminada.")
print(f"Archivos TXT encontrados: {archivos_txt:,}")
print(f"IDs únicos reconocidos y no vacíos: {len(ids_reconocidos):,}")
print(f"TXT con nombres no reconocidos: {archivos_no_reconocidos:,}")
print(f"Archivos vacíos: {archivos_vacios:,}")
print(f"Libros no procesados durante la selección: {errores_procesamiento:,}")
print(f"Documentos preparados: {len(documentos_tokens_grandes):,}")
print(f"Almacenamiento del TAR: {archivo_tar.stat().st_size / (1024 ** 3):.2f} GiB")

Leyendo el corpus grande...
Documentos procesados: 100
Documentos procesados: 200
Documentos procesados: 300
Documentos procesados: 400
Documentos procesados: 500
Documentos procesados: 600
Documentos procesados: 700
Documentos procesados: 800
Documentos procesados: 900
Documentos procesados: 1000

Lectura terminada.
Archivos TXT encontrados: 78,240
IDs únicos reconocidos y no vacíos: 78,240
TXT con nombres no reconocidos: 0
Archivos vacíos: 0
Libros no procesados durante la selección: 0
Documentos preparados: 1,000
Almacenamiento del TAR: 28.55 GiB


**7. Recuperación sobre el corpus grande**

Reutilice las funciones desarrolladas en la Parte A.

1. Procese los documentos con la misma configuración de normalización.
2. Ejecute las consultas mediante búsqueda secuencial y mida los tiempos.
3. Construya el índice invertido.
4. Repita las consultas utilizando el índice.
5. Compruebe la igualdad de resultados y compare los tiempos.

Trabaje con subconjuntos de la colección de tamaños crecientes, conservando los identificadores originales. Documente los recursos de memoria y almacenamiento utilizados.

In [91]:
import tracemalloc
tamanos = [50, 100, 250, 500, 1000]
consultas = ["the", "whale", "palabrainexistentexyz"]
identificadores = sorted(documentos_grandes.keys())

resultados = []

for n in tamanos:

    if n > len(identificadores):
        continue

    print(f"\nProcesando {n} documentos...")
    tracemalloc.start()
    documentos_procesados_grandes = {}

    for identificador in identificadores[:n]:
        texto = documentos_grandes[identificador]
        documentos_procesados_grandes[identificador] = preprocesar(texto)

    total_tokens = sum(
        len(tokens)
        for tokens in documentos_procesados_grandes.values()
    )
    inicio = time.perf_counter_ns()

    indice_grande = construir_indice_invertido(
        documentos_procesados_grandes
    )

    fin = time.perf_counter_ns()
    tiempo_construccion = (fin - inicio) / 1_000_000

    _, memoria_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    for consulta in consultas:

        tokens_consulta = preprocesar(consulta)

        if len(tokens_consulta) != 1:
            raise ValueError(
                f"La consulta {consulta!r} debe generar un solo término."
            )

        termino = tokens_consulta[0]
        inicio = time.perf_counter_ns()

        for _ in range(30):
            resultado_secuencial = busqueda_secuencial(
                termino,
                documentos_procesados_grandes
            )

        fin = time.perf_counter_ns()
        tiempo_secuencial = (fin - inicio) / 1_000_000 / 30
        inicio = time.perf_counter_ns()

        for _ in range(30):
            resultado_indice = busqueda_indice(
                termino,
                indice_grande
            )

        fin = time.perf_counter_ns()
        tiempo_indice = (fin - inicio) / 1_000_000 / 30
        iguales = set(resultado_secuencial) == set(resultado_indice)

        if not iguales:
            raise ValueError(
                f"Los resultados no coinciden para la consulta {termino!r}."
            )
        resultados.append({
            "Documentos": n,
            "Tokens": total_tokens,
            "Consulta": termino,
            "Encontrados": len(resultado_secuencial),
            "Secuencial (ms)": tiempo_secuencial,
            "Índice (ms)": tiempo_indice,
            "Iguales": iguales,
            "Construcción índice (ms)": tiempo_construccion,
            "Memoria pico (MiB)": memoria_pico / (1024 ** 2)
        })

tabla_resultados = pd.DataFrame(resultados)
display(tabla_resultados)

""


### Parte C. Análisis experimental de escalabilidad

**8. Tiempo de ejecución en función del corpus**

Seleccione al menos cinco tamaños crecientes del corpus grande. Como referencia, pueden utilizarse 50, 100, 250, 500 y 1.000 documentos, ampliándolos si el equipo lo permite.

Para cada tamaño $n$:

1. Ejecute una misma consulta mediante búsqueda secuencial.
2. Repítala 30 veces y calcule el tiempo promedio.
3. Construya el índice invertido correspondiente a ese subconjunto.
4. Ejecute la consulta mediante el índice, también 30 veces.
5. Registre ambos tiempos y el número total de tokens.

Elabore una **gráfica del tiempo medio de ejecución en función del número de documentos**, colocando el número de documentos en el eje X y el tiempo en milisegundos en el eje Y. Represente los dos métodos en la misma gráfica.

In [80]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [81]:
import matplotlib.pyplot as plt

consulta_elegida = "whale"
resultados_punto_8 = tabla_resultados[
    tabla_resultados["Consulta"] == consulta_elegida
].copy()
resultados_punto_8 = resultados_punto_8.sort_values("Documentos")
if len(resultados_punto_8) < 5:
    raise ValueError(
        "Se necesitan resultados para al menos cinco tamaños de corpus. "
        "Revisa que el punto 7 se haya ejecutado correctamente."
    )
display(
    resultados_punto_8[
        [
            "Documentos",
            "Tokens",
            "Ejecuciones",
            "Secuencial (ms)",
            "Índice (ms)",
            "Resultados iguales"
        ]
    ]
)

plt.figure(figsize=(10, 6))
plt.plot(
    resultados_punto_8["Documentos"],
    resultados_punto_8["Secuencial (ms)"],
    marker="o",
    label="Búsqueda secuencial"
)

plt.plot(
    resultados_punto_8["Documentos"],
    resultados_punto_8["Índice (ms)"],
    marker="s",
    label="Índice invertido"
)

plt.title(f"Tiempo medio de búsqueda para la consulta: '{consulta_elegida}'")
plt.xlabel("Número de documentos")
plt.ylabel("Tiempo medio de ejecución (ms)")

plt.xticks(resultados_punto_8["Documentos"])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

KeyError: 'Consulta'

**9. Estimación de una función de tiempo**

Utilizando los resultados experimentales, determine una función:

$t=f(n)$

para cada método de recuperación.

Como primera aproximación, utilice un ajuste lineal por mínimos cuadrados:

$t(n)=an+b$

Estime los parámetros $a$ y $b$, reporte el coeficiente de determinación $R^2$ y represente las funciones ajustadas junto a los datos experimentales.

Explique si el modelo lineal representa adecuadamente las mediciones y cómo se relaciona con la complejidad computacional de cada algoritmo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

n_documentos = resultados_punto_8["Documentos"].to_numpy(dtype=float)

tiempos_secuencial = resultados_punto_8["Secuencial (ms)"].to_numpy(dtype=float)

tiempos_indice = resultados_punto_8["Índice (ms)"].to_numpy(dtype=float)


def ajustar_recta(x, y):
    a, b = np.polyfit(x, y, 1)
    y_estimado = a * x + b
    suma_errores = np.sum((y - y_estimado) ** 2)
    suma_total = np.sum((y - np.mean(y)) ** 2)

    if np.isclose(suma_total, 0):
        r2 = np.nan
    else:
        r2 = 1 - (suma_errores / suma_total)

    return a, b, r2, y_estimado
a_sec, b_sec, r2_sec, estimados_sec = ajustar_recta(
    n_documentos,
    tiempos_secuencial
)

a_ind, b_ind, r2_ind, estimados_ind = ajustar_recta(
    n_documentos,
    tiempos_indice
)
tabla_ajustes = pd.DataFrame([
    {
        "Método": "Búsqueda secuencial",
        "Pendiente a": a_sec,
        "Intercepto b": b_sec,
        "R²": r2_sec
    },
    {
        "Método": "Índice invertido",
        "Pendiente a": a_ind,
        "Intercepto b": b_ind,
        "R²": r2_ind
    }
])

display(tabla_ajustes)

print("Funciones ajustadas:")
print(f"Secuencial: t(n) = {a_sec:.8f}n + ({b_sec:.8f}) ms")
print(f"Índice:     t(n) = {a_ind:.8f}n + ({b_ind:.8f}) ms")

print(f"\nR² secuencial: {r2_sec:.4f}")
print(f"R² índice:     {r2_ind:.4f}")
plt.figure(figsize=(10, 6))
plt.scatter(
    n_documentos,
    tiempos_secuencial,
    color="blue",
    label="Secuencial: mediciones"
)

plt.plot(
    n_documentos,
    estimados_sec,
    color="blue",
    linestyle="--",
    label="Secuencial: ajuste lineal"
)
plt.scatter(
    n_documentos,
    tiempos_indice,
    color="orange",
    label="Índice: mediciones"
)

plt.plot(
    n_documentos,
    estimados_ind,
    color="orange",
    linestyle="--",
    label="Índice: ajuste lineal"
)

plt.title("Tiempos experimentales y ajustes lineales")
plt.xlabel("Número de documentos")
plt.ylabel("Tiempo medio de ejecución (ms)")
plt.xticks(n_documentos)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()